# Full-Range Data Augmentation (0.1 – 100 µM)

This notebook extends the augmentation logic from `data_augmentation.ipynb` to cover the **complete experimental concentration interval**: `[0.1, 0.25, 0.5, 1, 2.5, 5, 7.5, 10, 15, 25, 50, 100] µM`.

### Key additions over the `[7.5, 20] µM` notebook

| Challenge | Solution |
|---|---|
| **Non-linearity at high concentrations (≥ 25 µM)** | Monotone cubic (PCHIP) spline fitted to empirical Ip values; per-segment dynamic α scaling |
| **Stratified oversampling of the low-concentration tail (< 2.5 µM)** | Concentration-proportional sample allocation with noise amplification |
| **Extended `get_base_pair` logic** | Covers all 12 anchor concentrations with automatic segment lookup |
| **Full `Signal` + `vectorize` compatibility** | Baseline-subtraction bypass identical to the parent notebook |

### Workflow
1. Load data & build all 12 mean base curves  
2. Characterise the Ip–concentration relationship (linearity check, PCHIP fit)  
3. Physics-informed augmentation: PCHIP interpolation + Gaussian noise + baseline variation + potential drift  
4. **Stratified sampling-oversampling of the low-concentration tail  
5. Route synthetic signals through the `Signal` + `vectorize` pipeline  
6. Persist combined CSVs for model training

In [5]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import PchipInterpolator
from scipy.signal import savgol_filter
from scipy.stats import linregress

# Plotly for interactive visualisation
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.preprocessing import StandardScaler

# project-specific classes
from voltammogram_signal import Signal
from peak import Peak

warnings.filterwarnings('ignore')
np.random.seed(42)

In [ ]:
# Reproducibility & shared Plotly styling
GLOBAL_RANDOM_SEED = 42
np.random.seed(GLOBAL_RANDOM_SEED)

# Default Plotly layout
PLOTLY_FIGURE_HEIGHT_PX = 520
PLOTLY_FIGURE_WIDTH_PX  = 880
PLOTLY_TEMPLATE         = 'plotly_white'


def apply_default_plotly_layout(figure: go.Figure,
                                title_text: str | None = None,
                                xaxis_title: str = 'Potential E (V)',
                                yaxis_title: str = 'Current I (µA)') -> go.Figure:
    """Apply the shared figure styling. Identical to the helper in
    data_augmentation.ipynb so figures look uniform across notebooks."""
    figure.update_layout(
        height=PLOTLY_FIGURE_HEIGHT_PX,
        width=PLOTLY_FIGURE_WIDTH_PX,
        template=PLOTLY_TEMPLATE,
        title=dict(text=title_text, x=0.5, xanchor='center') if title_text else None,
        xaxis_title=xaxis_title,
        yaxis_title=yaxis_title,
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1.0),
        margin=dict(l=60, r=30, t=70, b=50),
    )
    figure.update_xaxes(showgrid=True, minor=dict(showgrid=True))
    figure.update_yaxes(showgrid=True, minor=dict(showgrid=True))
    return figure


# A consistent colour palette for the 12 anchors (used throughout the notebook)
ANCHOR_COLOR_PALETTE = [
    '#2c0e91', '#3b0fa3', '#5a23c4', '#7b3fd6', '#9b5fe0',  # purples (low conc)
    '#b07ddb', '#c79bd2', '#e4b0a8', '#ee9070', '#f06d3a',  # transition
    '#e34a1a', '#b51d00',                                    # reds (high conc)
]
print('Plotly defaults configured.')


Plotly defaults configured.


### Load Data

Identical loading block to the parent notebook. All 12 experimental concentration levels are used as anchors.

In [ ]:
DATASET_PATH = 'datasets/Standard calibration in culture media_extended.xlsx'

raw_calibration_dataframe = pd.read_excel(DATASET_PATH, sheet_name='Raw data')

potential_grid_V        = raw_calibration_dataframe.iloc[1:, 0].values.astype(float)  # potential (V)
blank_baseline_current_uA  = raw_calibration_dataframe.iloc[1:, 1].values.astype(float)  # blank / baseline (µA)
raw_signal_matrix_uA     = raw_calibration_dataframe.iloc[1:, 2:].values.astype(float) # signal matrix (µA)

# Full concentration label list matching the 40 signal columns
CONCENTRATIONS = [
    100, 100, 100,
    50,  50,  50,
    25,  25,  25,
    15,  15,  15,
    10,  10,  10,
    7.5, 7.5, 7.5,
    5,   5,   5,
    2.5, 2.5,
    1,   1,   1,
    0.5, 0.5, 0.5, 0.5, 0.5, 0.5,
    0.25, 0.25, 0.25,
    0.1,  0.1,  0.1,  0.1,  0.1,
]

# All 12 unique anchor concentrations
ANCHOR_CONCENTRATIONS = [0.1, 0.25, 0.5, 1.0, 2.5, 5.0, 7.5, 10.0, 15.0, 25.0, 50.0, 100.0]

print(f"Potential range  : {potential_grid_V.min():.4f} V  →  {potential_grid_V.max():.4f} V")
print(f"Number of points : {len(potential_grid_V)}")
print(f"Signal columns   : {raw_signal_matrix_uA.shape[1]}")
print(f"Anchor concentrations: {ANCHOR_CONCENTRATIONS}")

Potential range  : -0.6001 V  →  0.5023 V
Number of points : 229
Signal columns   : 40
Anchor concentrations: [0.1, 0.25, 0.5, 1.0, 2.5, 5.0, 7.5, 10.0, 15.0, 25.0, 50.0, 100.0]


---
## Build Mean Base Curves for All 12 Anchors

For each anchor concentration we compute the mean baseline-subtracted signal across all replicates, then apply a Savitzky–Golay smooth to reduce inter-replicate noise before using it as an interpolation template.

In [ ]:
def compute_mean_baseline_subtracted_signal(signal_matrix_uA: np.ndarray,
                                            blank_baseline_uA: np.ndarray,
                                            concentration_labels_uM: list[float],
                                            target_concentration_uM: float) -> np.ndarray:
    """For one target concentration, return the mean of all baseline-subtracted
    replicates**. This gives a single high-SNR template curve at that anchor."""
    column_indices_for_target = [
        i for i, conc in enumerate(concentration_labels_uM)
        if conc == target_concentration_uM
    ]
    # subtract the blank from each replicate- keeps only the Faradaic signal
    baseline_subtracted_replicates = [
        signal_matrix_uA[:, i] - blank_baseline_uA
        for i in column_indices_for_target
    ]
    # average across replicates to suppress measurement noise
    return np.mean(baseline_subtracted_replicates, axis=0)

# Build the {concentration: mean_baseline_subtracted_signal} dictionary
base_curves = {
    target_c: compute_mean_baseline_subtracted_signal(
        raw_signal_matrix_uA, blank_baseline_current_uA,
        CONCENTRATIONS, target_c,
    )
    for target_c in ANCHOR_CONCENTRATIONS
}


# Report Ip for each anchor (smoothed before Peak detection)
print(f"{'Anchor (µM)':>12}  {'Ip (µA)':>10}  {'Ep (V)':>10}  {'Replicates':>10}")
print("-" * 50)
for c in ANCHOR_CONCENTRATIONS:
    n_reps = sum(1 for x in CONCENTRATIONS if x == c)
    pk = Peak(potential_grid_V, savgol_filter(base_curves[c], 5, 3))
    print(f"{c:>12}  {pk.Ip:>10.4f}  {pk.Ep:>10.4f}  {n_reps:>10}")

 Anchor (µM)     Ip (µA)      Ep (V)  Replicates
--------------------------------------------------
         0.1      0.0176     -0.3487           5
        0.25      0.0656     -0.3535           3
         0.5      0.1042     -0.3777           6
         1.0      0.2683     -0.3970           3
         2.5      0.7622     -0.3970           2
         5.0      1.6824     -0.3970           3
         7.5      2.7481     -0.3970           3
        10.0      4.0259     -0.3922           3
        15.0      6.1034     -0.3873           3
        25.0     11.2659     -0.3873           3
        50.0     22.7570     -0.3825           3
       100.0     34.2401     -0.3825           3


Interactive overview of all 12 anchor curves

In [10]:
anchor_overview_figure = go.Figure()

for anchor_concentration_uM, anchor_color in zip(ANCHOR_CONCENTRATIONS, ANCHOR_COLOR_PALETTE):
    anchor_overview_figure.add_trace(go.Scatter(
        x=potential_grid_V,
        y=base_curves[anchor_concentration_uM],
        mode='lines',
        name=f'Anchor {anchor_concentration_uM} µM',
        line=dict(color=anchor_color, width=2.2),
        hovertemplate=(f'{anchor_concentration_uM} µM<br>'
                       'E = %{x:.3f} V<br>'
                       'I = %{y:.4f} µA<extra></extra>'),
    ))

apply_default_plotly_layout(
    anchor_overview_figure,
    title_text='All 12 anchor curves (baseline-subtracted means)',
)
anchor_overview_figure.show()


---
## Characterising the Ip-Concentration Relationship

### Why Linear Interpolation Fails Above 25 µM

The Ip–concentration relationship is linear only at low concentrations. At higher concentrations, the relationship is no longer linear because the surface saturation effects reduce the marginal current per added molecule. (signal saturation)

The concrete data ilustrates:

| Segment | Δc | Expected ΔIp (linear) | Observed ΔIp ratio |
|---|---|---|---|
| 5 → 10 µM | ×2 | ×2 | ~×2 |
| 50 → 100 µM | ×2 | ×2 | ~×1.5 |

Linear interpolation in the 50–100 µM segment would systematically over-predict Ip, producing physically invalid synthetic records.

### Solution: PCHIP Spline on Empirical Ip Values

I try the Piecewise Cubic Hermite Interpolating Polynomial (PCHIP) to the 12 Ip values. PCHIP preserves monotonicity between data points (no oscillation artefacts) and captures the sub-linear rolloff at high concentrations.

The corrected interpolation weight `α_PCHIP` is derived from the PCHIP Ip–curve rather than from the raw concentration fraction:

$$\alpha_{\text{PCHIP}} = \frac{\hat{I}_p(c_{\text{target}}) - \hat{I}_p(c_{\text{low}})}{\hat{I}_p(c_{\text{high}}) - \hat{I}_p(c_{\text{low}})}$$

(pozitia relativa a valorii tinta in interiorul unui „segment” definit de doua puncte de calibrare cunoscute.)


This maps the concentration target to a current-space interpolation weight, so the blended curve has exactly the Ip the PCHIP model predicts (not what a naive linear α would produce - it would go in the middle if naive, but using PCHIP because of the saturation, la $15 \mu M$ curentul ar putea fi deja la $70\%$ din valoarea maximă a intervalului, nu la $50\%$).

For the 50–100 µM example with a target of 75 µM:

- Linear α = (75 − 50)/(100 − 50) = **0.50** → Ip_blend = 0.50 × Ip₅₀ + 0.50 × Ip₁₀₀  
- PCHIP α ≈ **0.40** (because the curve bends sub-linearly) → lower Ip blend, physically correct

In [ ]:
# Load the CC (calibration curve summary) sheet from the same workbook.
# It contains the **average peak signal per concentration** with stdev bars.
calibration_summary_df = pd.read_excel(DATASET_PATH, sheet_name='CC')

# Columns 10, 17 and 18 hold Concentration, Average Signal, Stdev (rows 4..15).
calibration_summary_df = calibration_summary_df.iloc[4:16, [10, 17, 18]].copy()
calibration_summary_df.columns = ['Concentration', 'Signal', 'Stdev']
calibration_summary_df = (calibration_summary_df
                          .apply(pd.to_numeric, errors='coerce')
                          .dropna()
                          .sort_values('Concentration')
                          .reset_index(drop=True))

# Split at the 25 µM saturation break-point
low_data  = calibration_summary_df[calibration_summary_df['Concentration'] <= 25.0]
high_data = calibration_summary_df[calibration_summary_df['Concentration'] >= 25.0]

# Two linear fits (one per regime)
slope_low,  intercept_low,  r_low,  *_ = linregress(low_data['Concentration'],  low_data['Signal'])
slope_high, intercept_high, r_high, *_ = linregress(high_data['Concentration'], high_data['Signal'])

# Build an interactive Plotly figure showing both regimes
dual_regime_figure = go.Figure()

# Empirical points with error bars (the actual measurements)
dual_regime_figure.add_trace(go.Scatter(
    x=calibration_summary_df['Concentration'],
    y=calibration_summary_df['Signal'],
    error_y=dict(type='data', array=calibration_summary_df['Stdev'], visible=True, color='black'),
    mode='markers', name='Empirical average Ip',
    marker=dict(color='black', size=10, symbol='circle',
                line=dict(color='white', width=1)),
    hovertemplate='c = %{x:.2f} µM<br>Ip = %{y:.4f} µA<extra></extra>',
))

# Linear-regime fit (blue, ≤ 25 µM)
x_low_fit = np.linspace(low_data['Concentration'].min(), 25, 50)
dual_regime_figure.add_trace(go.Scatter(
    x=x_low_fit, y=intercept_low + slope_low * x_low_fit,
    mode='lines',
    name=f'Linear regime (≤ 25 µM): y = {slope_low:.4f}x {intercept_low:+.4f}  (R² = {r_low**2:.4f})',
    line=dict(color='steelblue', width=2.5),
))

# Saturation-regime fit (red, ≥ 25 µM) 
dual_regime_figure.add_trace(go.Scatter(
    x=x_high_fit, y=intercept_high + slope_high * x_high_fit,
    mode='lines',
    name=f'Saturation regime (≥ 25 µM): y = {slope_high:.4f}x {intercept_high:+.4f}  (R² = {r_high**2:.4f})',
    line=dict(color='crimson', width=2.5, dash='dash'),
))

# Vertical guide at the break-point
dual_regime_figure.add_vline(
    x=25.0, line=dict(color='gray', dash='dot', width=1.2),
    annotation_text='break-point @ 25 µM', annotation_position='top',
)

apply_default_plotly_layout(
    dual_regime_figure,
    title_text='Dual-regime calibration: linear (≤ 25 µM) vs saturation (≥ 25 µM)',
    xaxis_title='Concentration (µM)',
    yaxis_title='Peak signal current Ip (µA)',
)
dual_regime_figure.show()

# Slope ratio is the headline number (quantifies how much the sensor "softens")
slope_ratio = slope_low / slope_high
print(f"Linear slope (≤ 25 µM): {slope_low:.4f} µA/µM   |   R² = {r_low**2:.4f}")
print(f"Saturated slope (≥ 25 µM): {slope_high:.4f} µA/µM  |   R² = {r_high**2:.4f}")
print(f"Slope ratio (low / high) = {slope_ratio:.2f}×   <-- this is the headline non-linearity")


Linear slope (≤ 25 µM): 0.4461 µA/µM   |   R² = 0.9947
Saturated slope (≥ 25 µM): 0.2966 µA/µM  |   R² = 0.9643
Slope ratio (low / high) = 1.50×   <-- this is the headline non-linearity


In [13]:
# Extract smoothed Ip values at each anchor
anchor_Ip = {}
for c in ANCHOR_CONCENTRATIONS:
    pk = Peak(potential_grid_V, savgol_filter(base_curves[c], 5, 3))
    anchor_Ip[c] = pk.Ip

# Pack into arrays for the spline fitter
c_arr  = np.array(ANCHOR_CONCENTRATIONS)
Ip_arr = np.array([anchor_Ip[c] for c in ANCHOR_CONCENTRATIONS])

# !!!!!!! Fit a PCHIP interpolant: Ip = f(concentration)
# PCHIP is preferred over CubicSpline because it cannotovershoot between knots,
# which is critical when extrapolating in the non-linear saturation regime.
pchip_Ip = PchipInterpolator(c_arr, Ip_arr)


# visualisation

# Ip–c relationship
c_fine = np.linspace(c_arr.min(), c_arr.max(), 500)
Ip_pchip_fine  = pchip_Ip(c_fine)
Ip_linear_fine = np.interp(c_fine, c_arr, Ip_arr)  # naive piecewise-linear, for comparison



In [16]:
# Figure of Full range, log x scale (so you can see the low-conc detail)
pchip_log_figure = go.Figure()

# Anchor stars (the ground truth)
pchip_log_figure.add_trace(go.Scatter(
    x=c_arr, y=Ip_arr,
    mode='markers', name='Empirical Ip at anchor',
    marker=dict(color='crimson', size=14, symbol='star',
                line=dict(color='black', width=1)),
    hovertemplate='c = %{x} µM<br>Ip = %{y:.4f} µA<extra></extra>',
))

# PCHIP - the physically-correct curve
pchip_log_figure.add_trace(go.Scatter(
    x=c_fine, y=Ip_pchip_fine,
    mode='lines', name='PCHIP spline (monotone cubic)',
    line=dict(color='steelblue', width=2.6),
    hovertemplate='c = %{x:.2f} µM<br>Ip_PCHIP = %{y:.4f} µA<extra></extra>',
))

# Naive linear (what i want to avoid above 25 µM)
pchip_log_figure.add_trace(go.Scatter(
    x=c_fine, y=Ip_linear_fine,
    mode='lines', name='Piecewise-linear (naive - over-predicts above 25 µM)',
    line=dict(color='darkorange', width=1.8, dash='dash'),
    hovertemplate='c = %{x:.2f} µM<br>Ip_linear = %{y:.4f} µA<extra></extra>',
))

pchip_log_figure.update_xaxes(type='log')
apply_default_plotly_layout(
    pchip_log_figure,
    title_text='Ip vs concentration on log scale (PCHIP vs linear)',
    xaxis_title='Concentration (µM, log scale)',
    yaxis_title='Peak current Ip (µA)',
)
pchip_log_figure.show()

In [ ]:
#  PCHIP vs linear gap
high_zoom_figure = go.Figure()
mask_hi = c_fine >= 20

high_zoom_figure.add_trace(go.Scatter(
    x=c_arr[c_arr >= 20], y=Ip_arr[c_arr >= 20],
    mode='markers', name='Empirical anchor',
    marker=dict(color='crimson', size=16, symbol='star',
                line=dict(color='black', width=1.2)),
))

high_zoom_figure.add_trace(go.Scatter(
    x=c_fine[mask_hi], y=Ip_pchip_fine[mask_hi],
    mode='lines', name='PCHIP (physically correct)',
    line=dict(color='steelblue', width=2.6),
))

high_zoom_figure.add_trace(go.Scatter(
    x=c_fine[mask_hi], y=Ip_linear_fine[mask_hi],
    mode='lines', name='Linear (over-predicts)',
    line=dict(color='darkorange', width=1.8, dash='dash'),
))

# Annotation at worst-case midpoints
for c_pt in [50, 75, 100]:
    Ip_p = float(pchip_Ip(c_pt))
    Ip_l = float(np.interp(c_pt, c_arr, Ip_arr))
    high_zoom_figure.add_annotation(
        x=c_pt, y=Ip_p,
        text=f'{c_pt} µM<br>PCHIP={Ip_p:.3f}<br>Linear={Ip_l:.3f}',
        showarrow=True, arrowhead=2, ax=20, ay=-50,
        font=dict(size=10), bgcolor='rgba(255,255,255,0.8)',
        bordercolor='gray', borderwidth=1, borderpad=4,
    )

apply_default_plotly_layout(
    high_zoom_figure,
    title_text=' Non-linear regime (≥ 20 µM): PCHIP vs linear gap',
    xaxis_title='Concentration (µM)',
    yaxis_title='Peak current Ip (µA)',
)
high_zoom_figure.show()

# Quantify divergence at the 75 µM midpoint
Ip_75_pchip  = float(pchip_Ip(75))
Ip_75_linear = float(np.interp(75, c_arr, Ip_arr))
print(f"At 75 µM (midpoint of 50–100 µM segment):")
print(f"  PCHIP Ip  = {Ip_75_pchip:.4f} µA")
print(f"  Linear Ip = {Ip_75_linear:.4f} µA")
print(f"  Over-prediction from linear = {(Ip_75_linear - Ip_75_pchip):.4f} µA "
      f"({(Ip_75_linear/Ip_75_pchip - 1)*100:.1f}%)")


At 75 µM (midpoint of 50–100 µM segment):
  PCHIP Ip  = 30.0094 µA
  Linear Ip = 28.4986 µA
  Over-prediction from linear = -1.5109 µA (-5.0%)


Zona $10^{-1}$ – $10^{1}$ ($0.1$ – $10 \mu M$ senzoruleste într-un regim liniar ideal. Creșterea curentului este direct proporțională cu concentrația.
Zona $10^{1}$ – $10^{2}$ ($10$ – $100 \mu M$): Curba începe să se „îndoaie” vizibil în sus pe scara logaritmică. Această pantă abruptă arată că semnalul crește puternic, distanța pe axa Y începe să se comprime spre final.

---
Augmentation Functions

`get_base_pair` Extended to All 12 Anchors

The lookup table now covers the full concentration axis. For any target `c`, we return the two adjacent anchor concentrations and their base curves.

In [ ]:
def get_base_pair(c_target: float):
    """
    Return (c_low, I_low, c_high, I_high) bracketing c_target across all 12 anchors.

    For concentrations at or below the minimum anchor (0.1 µM) or above the
    maximum (100 µM), the nearest boundary pair is used for extrapolation
    (conservative; avoids out-of-range spline behaviour).
    """
    # Guard: clamp to the observable range
    c_target = float(np.clip(c_target, ANCHOR_CONCENTRATIONS[0], ANCHOR_CONCENTRATIONS[-1]))

    for i in range(len(ANCHOR_CONCENTRATIONS) - 1):
        c_lo = ANCHOR_CONCENTRATIONS[i] # inferior limit
        c_hi = ANCHOR_CONCENTRATIONS[i + 1] # superior limit
        if c_lo <= c_target <= c_hi:
            return c_lo, base_curves[c_lo], c_hi, base_curves[c_hi]

    # Should never reach here after clamping, but fall back to last segment
    return ANCHOR_CONCENTRATIONS[-2], base_curves[ANCHOR_CONCENTRATIONS[-2]], ANCHOR_CONCENTRATIONS[-1], base_curves[ANCHOR_CONCENTRATIONS[-1]]


# Quick sanity check
for c_test in [0.15, 0.75, 3.0, 12.0, 30.0, 75.0]:
    c_lo, _, c_hi, _ = get_base_pair(c_test)
    print(f"  target = {c_test:6.2f} µM  →  segment [{c_lo}, {c_hi}] µM")

  target =   0.15 µM  →  segment [0.1, 0.25] µM
  target =   0.75 µM  →  segment [0.5, 1.0] µM
  target =   3.00 µM  →  segment [2.5, 5.0] µM
  target =  12.00 µM  →  segment [10.0, 15.0] µM
  target =  30.00 µM  →  segment [25.0, 50.0] µM
  target =  75.00 µM  →  segment [50.0, 100.0] µM


---
## 5 · Augmentation Functions


 1. `get_base_pair(c)` find the two anchor curves on either side of `c`.
 2. `interpolate_base_curve(c, ...)` blend them using the PCHIP-corrected α.
 3. `add_gaussian_noise / add_baseline_variation / apply_potential_drift` the three perturbations from the parent notebook (unchanged).
 4. `generate_synthetic_signal(c)` chain them all together for one signal.


In [ ]:
def get_base_pair(c_target: float):
    """
    Return (c_low, I_low, c_high, I_high) bracketing c_target across all 12 anchors.

    For concentrations at or below the minimum anchor (0.1 µM) or above the
    maximum (100 µM), the nearest boundary pair is used (conservative; avoids
    out-of-range spline behaviour).
    """
    # Guard: clamp to the observable range so the model never has to extrapolate
    c_target = float(np.clip(c_target, ANCHOR_CONCENTRATIONS[0], ANCHOR_CONCENTRATIONS[-1]))

    # Linear scan through the 12 anchors
    for i in range(len(ANCHOR_CONCENTRATIONS) - 1):
        c_lo = ANCHOR_CONCENTRATIONS[i]      # inferior limit of this segment
        c_hi = ANCHOR_CONCENTRATIONS[i + 1]  # superior limit of this segment
        if c_lo <= c_target <= c_hi:
            return c_lo, base_curves[c_lo], c_hi, base_curves[c_hi]

    # Should never reach here after clamping, but fall back to the last segment
    return (ANCHOR_CONCENTRATIONS[-2], base_curves[ANCHOR_CONCENTRATIONS[-2]],
            ANCHOR_CONCENTRATIONS[-1], base_curves[ANCHOR_CONCENTRATIONS[-1]])


# every target should land in a segment that contains it
print("Segment lookup sanity check:")
for c_test in [0.15, 0.75, 3.0, 12.0, 30.0, 75.0]:
    c_lo, _, c_hi, _ = get_base_pair(c_test)
    print(f"  target = {c_test:6.2f} µM  →  segment [{c_lo}, {c_hi}] µM")


Segment lookup sanity check:
  target =   0.15 µM  →  segment [0.1, 0.25] µM
  target =   0.75 µM  →  segment [0.5, 1.0] µM
  target =   3.00 µM  →  segment [2.5, 5.0] µM
  target =  12.00 µM  →  segment [10.0, 15.0] µM
  target =  30.00 µM  →  segment [25.0, 50.0] µM
  target =  75.00 µM  →  segment [50.0, 100.0] µM


### PCHIP-corrected Interpolation

Standard linear blend with the PCHIP-corrected alpha weight.

**For segments ≤ 25 µM**: the PCHIP spline closely follows a linear trend (Faradaic regime), so the corrected alpha barely differs from `(c_target − c_low) / (c_high − c_low)`. The function is safe to use uniformly. PCHIP α ≈ linear α (Faradaic regime). Safe.

**For segments > 25 µM**: the corrected alpha is noticeably smaller than the linear alpha, pulling the blended curve's Ip down to the physically correct value.

In [ ]:
def interpolate_base_curve(c_target: float, c_low: float, I_low: np.ndarray,
                           c_high: float, I_high: np.ndarray, use_pchip: bool = True) -> np.ndarray:
    """
    Blend two base curves weighted by the PCHIP-corrected alpha.
    Alpha is computed in current-space using the PCHIP Ip spline:
        alpha_PCHIP = (Ip_hat(c_target) - Ip_hat(c_low)) / (Ip_hat(c_high) - Ip_hat(c_low))
    This ensures the blended curve has the physically correct peak current for
    all segments, especially the non-linear high-concentration regime.

    -use_pchip=True (default):  α is derived in CURRENT space via the PCHIP
      Ip spline, so the blended curve's Ip exactly matches the physically
      correct value at c_target. This is the main full-range innovation.

    -use_pchip=False:  fall back to the naive concentration-space α from the
      parent notebook. Kept as an option only for the ablation study (so we can
      quantify how much PCHIP actually gets us).


    """
    if c_high == c_low:
        return I_low.copy()

    if use_pchip:
        # PCHIP alpha ( measured in CURRENT space)
        Ip_low_hat    = float(pchip_Ip(c_low))
        Ip_high_hat   = float(pchip_Ip(c_high))
        Ip_target_hat = float(pchip_Ip(c_target))
        denom         = Ip_high_hat - Ip_low_hat
        if abs(denom) < 1e-9:
            # identical Ip at the two anchors -> fall back to concentration α
            alpha = (c_target - c_low) / (c_high - c_low)
        else:
            alpha = (Ip_target_hat - Ip_low_hat) / denom
    else:
        # naive linear alpha in concentration space only
        alpha = (c_target - c_low) / (c_high - c_low)


    alpha = float(np.clip(alpha, 0.0, 1.0))  # numerical safety
    return (1.0 - alpha) * I_low + alpha * I_high


# Demonstrate alpha correction for the 50-100 µM segment
print("Alpha comparison on 50 -> 100 µM segment")
print(f"{'c_target':>10}  {'alpha_linear':>13}  {'alpha_PCHIP':>12}  {'Ip_PCHIP':>10}")
print("-" * 52)
for c_t in [55, 60, 65, 70, 75, 80, 85, 90, 95]:
    alpha_lin   = (c_t - 50) / (100 - 50)
    Ip_lo       = float(pchip_Ip(50))
    Ip_hi       = float(pchip_Ip(100))
    # curentul pe care senzorul ar trebui sa
    # il produca daca fenomenul de saturatie a piocianinei 
    # urmeaza traiectoria curba determinata de restul punctelor masurate
    Ip_t        = float(pchip_Ip(c_t)) 
    alpha_pchip = (Ip_t - Ip_lo) / (Ip_hi - Ip_lo)
    delta_alpha = alpha_pchip - alpha_lin
    print(f"{c_t:>10}  {alpha_lin:>13.4f}  {alpha_pchip:>12.4f}  {Ip_t:>10.4f} {delta_alpha:>+8.4f}")

Alpha comparison on 50 -> 100 µM segment
  c_target   alpha_linear   alpha_PCHIP    Ip_PCHIP
----------------------------------------------------
        55         0.1000        0.1372     24.3324  +0.0372
        60         0.2000        0.2706     25.8648  +0.0706
        65         0.3000        0.3987     27.3348  +0.0987
        70         0.4000        0.5195     28.7228  +0.1195
        75         0.5000        0.6316     30.0094  +0.1316
        80         0.6000        0.7331     31.1752  +0.1331
        85         0.7000        0.8224     32.2006  +0.1224
        90         0.8000        0.8978     33.0662  +0.0978
        95         0.9000        0.9575     33.7525  +0.0575


In [32]:
# Visualise α_PCHIP vs α_linear across the fullL range
alpha_compare_figure = go.Figure()

# Sample α_PCHIP at fine resolution within every segment
c_targets   = np.linspace(ANCHOR_CONCENTRATIONS[0] + 1e-3,
                          ANCHOR_CONCENTRATIONS[-1] - 1e-3, 400)
alpha_linear_arr = np.empty_like(c_targets)
alpha_pchip_arr  = np.empty_like(c_targets)
for k, c_t in enumerate(c_targets):
    c_lo, _, c_hi, _ = get_base_pair(c_t)
    alpha_linear_arr[k] = (c_t - c_lo) / (c_hi - c_lo)
    Ip_lo = float(pchip_Ip(c_lo)); Ip_hi = float(pchip_Ip(c_hi))
    if abs(Ip_hi - Ip_lo) < 1e-9:
        alpha_pchip_arr[k] = alpha_linear_arr[k]
    else:
        alpha_pchip_arr[k] = (float(pchip_Ip(c_t)) - Ip_lo) / (Ip_hi - Ip_lo)

alpha_compare_figure.add_trace(go.Scatter(
    x=c_targets, y=alpha_linear_arr,
    mode='lines', name='α_linear (within segment)',
    line=dict(color='darkorange', width=2.2, dash='dash'),
    hovertemplate='c = %{x:.2f} µM<br>α_linear = %{y:.3f}<extra></extra>',
))
alpha_compare_figure.add_trace(go.Scatter(
    x=c_targets, y=alpha_pchip_arr,
    mode='lines', name='α_PCHIP (within segment)',
    line=dict(color='steelblue', width=2.6),
    hovertemplate='c = %{x:.2f} µM<br>α_PCHIP = %{y:.3f}<extra></extra>',
))

# Mark anchor boundaries
for ac in ANCHOR_CONCENTRATIONS:
    alpha_compare_figure.add_vline(x=ac, line=dict(color='lightgray', width=0.8))

apply_default_plotly_layout(
    alpha_compare_figure,
    title_text='α_PCHIP vs α_linear within each segment',
    xaxis_title='Target concentration (µM)',
    yaxis_title='Interpolation weight α',
)
alpha_compare_figure.update_xaxes(type='log')
alpha_compare_figure.show()


### Noise, Baseline Variation, and Potential Drift

Identical implementations to the parent notebook, unchanged.

In [33]:
DEFAULT_INSTRUMENT_NOISE_SIGMA_uA   = 0.001   # µA
BASELINE_VARIATION_MAX_AMPLITUDE_uA = 0.05    # µA

POTENTIAL_DRIFT_SIGMA_LOW_V         = 0.01    # V
POTENTIAL_DRIFT_SIGMA_HIGH_V        = 0.03    # V


DEFAULT_INSTRUMENT_NOISE_SIGMA_uA = 0.001  # original value

def apply_gaussian_instrument_noise(
    clean_current_signal: np.ndarray,
    noise_sigma_uA: float = DEFAULT_INSTRUMENT_NOISE_SIGMA_uA,
) -> np.ndarray:
    """Add Gaussian noise of standard deviation `noise_sigma_uA` µA to
    every sample of the input signal."""
    additive_noise = np.random.normal(loc=0.0, scale=noise_sigma_uA,
                                       size=clean_current_signal.shape)
    return clean_current_signal + additive_noise


BASELINE_VARIATION_MAX_AMPLITUDE_uA = 0.05

def apply_polynomial_baseline_distortion(
    clean_current_signal: np.ndarray,
    potential_grid_V: np.ndarray,
    max_amplitude_uA: float = BASELINE_VARIATION_MAX_AMPLITUDE_uA,
) -> np.ndarray:
    """Add a smooth `amp * (E_norm² - E_norm⁴)` polynomial bump to the signal.
    `amp` is drawn from U(-max_amplitude_uA, +max_amplitude_uA) so the distortion
    can be a bump or a dip with equal probability.
    """
    distortion_amplitude_uA = np.random.uniform(-max_amplitude_uA, +max_amplitude_uA)
    # normalisation
    normalised_potential    = (
        (potential_grid_V - potential_grid_V.mean())
        / (potential_grid_V.max() - potential_grid_V.min())
    )
    # distortion
    distortion_curve_uA     = (
        distortion_amplitude_uA
        * (normalised_potential ** 2 - normalised_potential ** 4)
    )
    # return the distorted signal
    return clean_current_signal + distortion_curve_uA

def apply_horizontal_potential_drift(
    clean_current_signal: np.ndarray,
    potential_grid_V: np.ndarray,
    drift_sigma_low_V:  float = POTENTIAL_DRIFT_SIGMA_LOW_V,
    drift_sigma_high_V: float = POTENTIAL_DRIFT_SIGMA_HIGH_V,
) -> np.ndarray:
    """Shift the signal by ΔE ~ N(0, σ²) along the potential axis, then linearly
    re-interpolate back onto the original grid (endpoints held constant)."""
    drift_sigma_V    = np.random.uniform(drift_sigma_low_V, drift_sigma_high_V)
    delta_potential_V = np.random.normal(loc=0.0, scale=drift_sigma_V)
    shifted_potential_grid_V = potential_grid_V + delta_potential_V
    return np.interp(
        potential_grid_V, shifted_potential_grid_V, clean_current_signal,
        left=clean_current_signal[0], right=clean_current_signal[-1],
    )

def generate_synthetic_signal(c_target: float, use_pchip: bool = True) -> np.ndarray:
    """Full augmentation pipeline: interpolate → noise → baseline → drift."""
    c_low, I_low, c_high, I_high = get_base_pair(c_target)
    I = interpolate_base_curve(c_target, c_low, I_low, c_high, I_high, use_pchip=use_pchip)
    I = apply_gaussian_instrument_noise(I)
    I = apply_polynomial_baseline_distortion (I, potential_grid_V)
    I = apply_horizontal_potential_drift(I, potential_grid_V)
    return I



print("Augmentation helpers ready.")


Augmentation helpers ready.


---
## Stratified Sampling Strategy -> FOR THE LOW CONCENTRATION TAIL

* WIP *


In [ ]:
# TODO: understand lower end of the concentration range 
# and figure out if it requires an allocation strategy

## 7 · Generate the Full Synthetic Dataset

In [36]:
def generate_flat_high_end_dataset(N_total: int,
                                   use_pchip: bool = True,
                                   noise_sigma_const_uA: float = DEFAULT_INSTRUMENT_NOISE_SIGMA_uA,
                                   baseline_amp_uA: float = BASELINE_VARIATION_MAX_AMPLITUDE_uA,
                                   enable_drift: bool = True,
                                   rng_seed: int = GLOBAL_RANDOM_SEED):
    """
    Generate synthetic pairs using a flat allocation exclusively for the high-end range.
    Bypasses inverse-log stratification and low-concentration Lorentzian noise.
    """
    np.random.seed(rng_seed) 
    aug_concentrations = []
    aug_signals_I      = []

    # Flatten the allocation and restrict bounds to [7.5, 100]
    c_targets = np.random.uniform(7.5, 100.0, size=N_total)
    
    for c_t in c_targets:
        # Interpolate the base curve
        c_low, I_low, c_high, I_high = get_base_pair(c_t)
        I = interpolate_base_curve(c_t, c_low, I_low, c_high, I_high, use_pchip=use_pchip)

        # 2. Apply standard perturbations
        if noise_sigma_const_uA > 0:
            I = apply_gaussian_instrument_noise(I, noise_sigma_uA=noise_sigma_const_uA)
        if baseline_amp_uA > 0:
            I = apply_polynomial_baseline_distortion(I, potential_grid_V, max_amplitude_uA=baseline_amp_uA)
        if enable_drift:
            I = apply_horizontal_potential_drift(I, potential_grid_V)

        aug_concentrations.append(c_t)
        aug_signals_I.append(I)

    # Sort by concentration
    order = np.argsort(aug_concentrations)
    return (np.array(aug_concentrations)[order],
            [aug_signals_I[k] for k in order])

# Run test
aug_concentrations, aug_signals_I = generate_flat_high_end_dataset(N_total=300)

print(f"Generated {len(aug_signals_I)} synthetic signals")
print(f"Concentration range: {aug_concentrations.min():.3f} – {aug_concentrations.max():.3f} µM")

Generated 300 synthetic signals
Concentration range: 7.968 – 99.080 µM


## 8 · Inspection of the Synthetic Dataset

### 8.1 · Signal gallery representative curves across the full range


In [57]:
# 4 representative concentration zones for the restricted [7.5, 100] µM range
inspection_zones_high = [
    (7.5,  15.0,  'Mid-high (7.5–15 µM)'),
    (15.0, 25.0,  'Pre-saturation (15–25 µM)'),
    (25.0, 50.0,  'Saturation onset (25–50 µM)'),
    (50.0, 100.0, 'Deep saturation (50–100 µM)'),
]

signal_gallery_figure = make_subplots(
    rows=2, cols=2, # Changed to a 2x2 grid
    subplot_titles=[z[2] for z in inspection_zones_high],
    horizontal_spacing=0.08, vertical_spacing=0.18,
)

for panel_idx, (c_lo, c_hi, _title) in enumerate(inspection_zones_high):
    # Updated grid math for 2 columns instead of 3
    row = panel_idx // 2 + 1
    col = panel_idx %  2 + 1

    # Pick up to 6 synthetic signals from this zone
    mask = (aug_concentrations >= c_lo) & (aug_concentrations <= c_hi)
    idxs = np.where(mask)[0]
    if len(idxs) == 0:
        continue
    n_show       = min(6, len(idxs))
    sample_idxs  = np.linspace(0, len(idxs) - 1, n_show, dtype=int)

    # Colour synthetic curves by concentration within the zone
    for j, si in enumerate(sample_idxs):
        real_idx = idxs[si]
        # Use HSL gradient for clear concentration ordering within the panel
        h = int(240 - 240 * (j / max(n_show - 1, 1)))
        signal_gallery_figure.add_trace(go.Scatter(
            x=potential_grid_V, y=aug_signals_I[real_idx],
            mode='lines', name=f'{aug_concentrations[real_idx]:.2f} µM',
            line=dict(color=f'hsl({h}, 70%, 45%)', width=1.1),
            opacity=0.75, showlegend=False,
            hovertemplate=(f'Synth {aug_concentrations[real_idx]:.2f} µM<br>'
                           'E = %{x:.3f} V<br>I = %{y:.4f} µA<extra></extra>'),
        ), row=row, col=col)

    # Overlay the anchor base curves in this zone (black dashed)
    for c_anc in ANCHOR_CONCENTRATIONS:
        if c_lo <= c_anc <= c_hi:
            signal_gallery_figure.add_trace(go.Scatter(
                x=potential_grid_V, y=base_curves[c_anc],
                mode='lines', name=f'{c_anc} µM anchor',
                line=dict(color='black', width=1.6, dash='dash'),
                opacity=0.8, showlegend=False,
                hovertemplate=f'Anchor {c_anc} µM<extra></extra>',
            ), row=row, col=col)

# Constrain x-axes to where the peak actually sits
for r in (1, 2):
    for c in (1, 2): # Updated to loop only over 2 columns
        signal_gallery_figure.update_xaxes(range=[-0.65, 0.10], row=r, col=c,
                                           showgrid=True, minor=dict(showgrid=True))
        signal_gallery_figure.update_yaxes(showgrid=True, minor=dict(showgrid=True),
                                           row=r, col=c)

signal_gallery_figure.update_layout(
    height=650, width=950,
    template=PLOTLY_TEMPLATE,
    title=dict(text='Synthetic signal gallery (High-end subset: 7.5–100 µM)',
               x=0.5, xanchor='center'),
    margin=dict(l=60, r=30, t=80, b=50),
    showlegend=False,
)
signal_gallery_figure.show()

### 8.2 · Ip vs concentration..  does the synthetic dataset track the PCHIP target?

I re-measure `Ip` from every synthetic curve and plot it against its target concentration. The cloud of synthetic points should hug the orange PCHIP target curve (with some scatter from the added noise), not the dashed naive-linear line.


In [59]:
# Re-measure Ip from every synthetic curve (after a mild Savitzky-Golay smooth)
Ip_synth = []
for I_s in aug_signals_I:
    try:
        pk = Peak(potential_grid_V, savgol_filter(I_s, 5, 3))
        Ip_synth.append(pk.Ip)
    except Exception:
        Ip_synth.append(np.nan)
Ip_synth = np.array(Ip_synth)
valid    = ~np.isnan(Ip_synth)

# Full range, log-x scale
ip_check_figure = go.Figure()

ip_check_figure.add_trace(go.Scatter(
    x=aug_concentrations[valid], y=Ip_synth[valid],
    mode='markers', name='Synthetic Ip',
    marker=dict(color='steelblue', size=5, opacity=0.45,
                line=dict(color='steelblue', width=0)),
    hovertemplate='c = %{x:.3f} µM<br>Ip = %{y:.4f} µA<extra></extra>',
))
ip_check_figure.add_trace(go.Scatter(
    x=c_fine, y=pchip_Ip(c_fine),
    mode='lines', name='PCHIP target',
    line=dict(color='darkorange', width=2.4),
))
ip_check_figure.add_trace(go.Scatter(
    x=c_fine, y=np.interp(c_fine, c_arr, Ip_arr),
    mode='lines', name='Naive linear (reference, NOT used)',
    line=dict(color='gray', width=1.2, dash='dot'),
))
ip_check_figure.add_trace(go.Scatter(
    x=c_arr, y=Ip_arr,
    mode='markers', name='Empirical anchor',
    marker=dict(color='crimson', size=14, symbol='star',
                line=dict(color='black', width=1)),
))

ip_check_figure.update_xaxes(type='log')
apply_default_plotly_layout(
    ip_check_figure,
    title_text='Synthetic Ip vs PCHIP target (full range, log scale)',
    xaxis_title='Concentration (µM, log scale)',
    yaxis_title='Peak current Ip (µA)',
)
ip_check_figure.show()


# high-conc zoom (linear scale)
ip_zoom_figure = go.Figure()

mask_hi  = aug_concentrations >= 20
ip_zoom_figure.add_trace(go.Scatter(
    x=aug_concentrations[mask_hi & valid], y=Ip_synth[mask_hi & valid],
    mode='markers', name='Synthetic Ip (≥ 20 µM)',
    marker=dict(color='steelblue', size=8, opacity=0.55,
                line=dict(color='black', width=0.4)),
))
ip_zoom_figure.add_trace(go.Scatter(
    x=c_fine[c_fine >= 20], y=pchip_Ip(c_fine)[c_fine >= 20],
    mode='lines', name='PCHIP target',
    line=dict(color='darkorange', width=2.6),
))
ip_zoom_figure.add_trace(go.Scatter(
    x=c_fine[c_fine >= 20], y=np.interp(c_fine, c_arr, Ip_arr)[c_fine >= 20],
    mode='lines', name='Naive linear (would over-predict)',
    line=dict(color='gray', width=1.4, dash='dot'),
))
ip_zoom_figure.add_trace(go.Scatter(
    x=c_arr[c_arr >= 20], y=Ip_arr[c_arr >= 20],
    mode='markers', name='Empirical anchor',
    marker=dict(color='crimson', size=14, symbol='star',
                line=dict(color='black', width=1)),
))
apply_default_plotly_layout(
    ip_zoom_figure,
    title_text='Saturation-regime zoom (≥ 20 µM): synthetic Ip vs PCHIP target',
    xaxis_title='Concentration (µM)',
    yaxis_title='Peak current Ip (µA)',
)
ip_zoom_figure.show()

# Per-segment residuals: how close is the median synthetic Ip to the PCHIP target?
print("Per-segment residual check (median synthetic Ip vs PCHIP target):")
print(f"{'Segment':>20}  {'PCHIP Ip':>10}  {'Median syn':>12}  {'Residual %':>12}")
print('-' * 60)
for i in range(len(ANCHOR_CONCENTRATIONS) - 1):
    c_lo = ANCHOR_CONCENTRATIONS[i]; c_hi = ANCHOR_CONCENTRATIONS[i + 1]
    seg_mask = (aug_concentrations >= c_lo) & (aug_concentrations < c_hi) & valid
    if seg_mask.sum() == 0: continue
    c_mid      = (c_lo + c_hi) / 2
    Ip_target  = float(pchip_Ip(c_mid))
    Ip_median  = float(np.median(Ip_synth[seg_mask]))
    residual   = (Ip_median - Ip_target) / Ip_target * 100
    print(f"{f'{c_lo}–{c_hi} µM':>20}  {Ip_target:>10.4f}  {Ip_median:>12.4f}  {residual:>+11.2f}%")


Per-segment residual check (median synthetic Ip vs PCHIP target):
             Segment    PCHIP Ip    Median syn    Residual %
------------------------------------------------------------
         7.5–10.0 µM      3.3874        3.4694        +2.42%
        10.0–15.0 µM      5.0701        4.7561        -6.19%
        15.0–25.0 µM      8.6404        8.7548        +1.32%
        25.0–50.0 µM     17.5499       16.8979        -3.72%
       50.0–100.0 µM     30.0094       29.8746        -0.45%


In [42]:
def plot_real_vs_synthetic_comparison(test_concs: list[float] = (0.5, 10.0, 75.0),
                                       n_synthetic_per_panel: int = 15) -> go.Figure:
    """Overlay experimental replicates and a batch of synthetic signals."""
    fig = make_subplots(
        rows=1, cols=len(test_concs),
        subplot_titles=[f'c = {c} µM' for c in test_concs],
        horizontal_spacing=0.07,
    )

    np.random.seed(7)  # deterministic for the figure caption stats

    for panel_idx, c_t in enumerate(test_concs):
        col = panel_idx + 1

        # 1) Real measurements at this concentration (baseline-subtracted)
        real_idxs = [i for i, c in enumerate(CONCENTRATIONS) if c == c_t]
        for k, idx in enumerate(real_idxs):
            I_real = raw_signal_matrix_uA[:, idx] - blank_baseline_current_uA
            fig.add_trace(go.Scatter(
                x=potential_grid_V, y=I_real,
                mode='lines',
                name='Real replicate' if (k == 0 and panel_idx == 0) else None,
                line=dict(color='black', width=1.8), opacity=0.85,
                showlegend=(k == 0 and panel_idx == 0),
                hovertemplate=f'Real {c_t} µM<br>E = %{{x:.3f}}<br>I = %{{y:.4f}}<extra></extra>',
            ), row=1, col=col)

        # 2) A batch of synthetic curves at exactly this concentration
        for k in range(n_synthetic_per_panel):
            I_syn = generate_synthetic_signal(c_t)
            fig.add_trace(go.Scatter(
                x=potential_grid_V, y=I_syn,
                mode='lines',
                name='Synthetic' if (k == 0 and panel_idx == 0) else None,
                line=dict(color='#17becf', width=0.9), opacity=0.4,
                showlegend=(k == 0 and panel_idx == 0),
                hovertemplate=f'Synth {c_t} µM<br>E = %{{x:.3f}}<br>I = %{{y:.4f}}<extra></extra>',
            ), row=1, col=col)

        fig.update_xaxes(title_text='Potential E (V)', row=1, col=col,
                          showgrid=True, minor=dict(showgrid=True))
        fig.update_yaxes(title_text='Current I (µA)' if col == 1 else None,
                          row=1, col=col, showgrid=True, minor=dict(showgrid=True))

    fig.update_layout(
        height=PLOTLY_FIGURE_HEIGHT_PX, width=1200, template=PLOTLY_TEMPLATE,
        title=dict(text='Real replicates vs synthetic signals (low / mid / high)',
                   x=0.5, xanchor='center'),
        legend=dict(orientation='h', yanchor='bottom', y=1.06, xanchor='right', x=1.0),
        margin=dict(l=60, r=30, t=90, b=60),
    )
    return fig


plot_real_vs_synthetic_comparison(test_concs=[0.5, 10.0, 75.0]).show()


## 9 · Feature Extraction: Routing Through the `Signal` + `vectorize` Pipeline

In [63]:
def vectorize_signal_features(signal_object: 'Signal', feature_suite: str = 'core') -> list:
    """Extract a flat feature vector from a Signal object for one of the
    three named suites ('core' / 'extended' / 'experimental')."""
    feature_vector = []
    if feature_suite in ('core', 'extended', 'experimental'):
        feature_vector += [
            signal_object.get_peak_current_value(),
            signal_object.get_peak_potential_value(),
            signal_object.get_peak_auc(),
            signal_object.get_peak_fwhm(),
        ]
    if feature_suite in ('extended', 'experimental'):
        feature_vector += [
            signal_object.get_pca1_comp(),
            signal_object.get_first_derivative_max(),
            signal_object.get_second_derivative_min(),
        ]
    if feature_suite == 'experimental':
        feature_vector += [
            signal_object.get_left_slope(),
            signal_object.get_right_slope(),
            signal_object.get_asymetry(),
            signal_object.get_peak_sharpness(),
            signal_object.get_peak_compactness(),
            signal_object.get_current_variance(),
            signal_object.get_peak_skewness(),
            signal_object.get_peak_kurtosis(),
            signal_object.get_tchebichef_curve_moments(),
            signal_object.get_mean_peak(),
            signal_object.get_signal_entropy(),
            signal_object.get_spectral_entropy(),
            signal_object.get_fft_power(),
            signal_object.get_pca2_comp(),
            signal_object.get_pca3_comp(),
            signal_object.get_wavelet_energy(),
        ]
    return feature_vector


FEATURE_COLUMN_NAMES_BY_SUITE = {
    'core':     ['peak_current', 'peak_potential', 'peak_AUC', 'peak_FWHM'],
    'extended': ['peak_current', 'peak_potential', 'peak_AUC', 'peak_FWHM',
                 'pca1_comp', 'first_derivative_max', 'second_derivative_min'],
    'experimental': ['peak_current', 'peak_potential', 'peak_AUC', 'peak_FWHM',
                     'pca1_comp', 'first_derivative_max', 'second_derivative_min',
                     'left_slope', 'right_slope', 'asymetry', 'peak_sharpness',
                     'peak_compactness', 'current_variance', 'peak_skewness',
                     'peak_kurtosis', 'tchebichef_curve_moments', 'mean_peak',
                     'signal_entropy', 'spectral_entropy', 'fft_power',
                     'pca2_comp', 'pca3_comp', 'wavelet_energy'],
}


In [64]:
# Process ORIGINAL signals (baseline subtraction handled by Signal automatically)
Signal.set_common_potential_E(potential_grid_V)
Signal.set_common_baseline_I(blank_baseline_current_uA)
Signal.sig_id = 1

original_signals_with_labels = []
for column_index in range(raw_signal_matrix_uA.shape[1]):
    try:
        sig = Signal(raw_signal_matrix_uA[:, column_index])
        original_signals_with_labels.append((sig, CONCENTRATIONS[column_index]))
    except Exception as e:
        print(f"  [WARN] original signal {column_index} skipped: {e}")

print(f"Original signals processed: {len(original_signals_with_labels)}")

Original signals processed: 40


In [65]:
# Process AUGMENTED signals
# Augmented signals are already baseline-subtracted; disable auto-subtraction.

def process_synthetic_currents_into_signal_objects(
        synthetic_current_signals: list[np.ndarray],
        synthetic_target_concentrations_uM: np.ndarray,
        ) -> tuple[list[tuple['Signal', float]], int]:
    
    # setting to [] reduces errors from synthetic signals that have baseline distortions exceeding the original blank baseline
    Signal.set_common_baseline_I(np.array([]))
    signal_objects_with_labels = []
    
    num_skipped = 0
    for synthetic_current_signal, target_concentration_uM in zip(
        synthetic_current_signals, synthetic_target_concentrations_uM
    ):
        try:
            signal_objects_with_labels.append(
                (Signal(synthetic_current_signal), float(target_concentration_uM))
            )
        except Exception:
            num_skipped += 1
    Signal.set_common_baseline_I(blank_baseline_current_uA) # restore
    return signal_objects_with_labels, num_skipped

default_synthetic_signal_objects, default_num_skipped = (
    process_synthetic_currents_into_signal_objects(
        aug_signals_I, aug_concentrations
    )
)
print(f'Default synthetic signals processed: '
      f'{len(default_synthetic_signal_objects)}  (skipped: {default_num_skipped})')


Default synthetic signals processed: 300  (skipped: 0)


In [ ]:
def build_feature_dataframe(signal_objects_with_labels: list,
                            feature_suite: str) -> pd.DataFrame:
    """Vectorise every signal and assemble a DataFrame whose last column is the
    concentration target."""
    feature_rows = []
    num_skipped  = 0
    for signal_obj, target_concentration_uM in signal_objects_with_labels:
        try:
            feature_rows.append(
                vectorize_signal_features(signal_obj, feature_suite)
                + [target_concentration_uM]
            )
        except Exception:
            num_skipped += 1
    if num_skipped > 0:
        print(f">>> [{feature_suite}] Skipped {num_skipped} signals due to extraction errors.")
    return pd.DataFrame(
        feature_rows,
        columns=FEATURE_COLUMN_NAMES_BY_SUITE[feature_suite] + ['concentration'],
    )


# Build the six DataFrames: 3 suites × {originals, augmented}
orig_core = build_feature_dataframe(original_signals_with_labels, 'core')
orig_ext  = build_feature_dataframe(original_signals_with_labels, 'extended')
orig_exp  = build_feature_dataframe(original_signals_with_labels, 'experimental')

aug_core  = build_feature_dataframe(default_synthetic_signal_objects, 'core')
aug_ext   = build_feature_dataframe(default_synthetic_signal_objects, 'extended')
aug_exp   = build_feature_dataframe(default_synthetic_signal_objects, 'experimental')

# Combined (original + augmented) the dataset to use for Protocol-C training
combined_core = pd.concat([orig_core, aug_core], ignore_index=True)
combined_ext  = pd.concat([orig_ext,  aug_ext],  ignore_index=True)
combined_exp  = pd.concat([orig_exp,  aug_exp],  ignore_index=True)

print(f"Original - core: {orig_core.shape}  | extended: {orig_ext.shape}  | experimental: {orig_exp.shape}")
print(f"Augmented -  core: {aug_core.shape}   | extended: {aug_ext.shape}    | experimental: {aug_exp.shape}")
print(f"Combined - core: {combined_core.shape} | extended: {combined_ext.shape} | experimental: {combined_exp.shape}")


>>> [core] Skipped 105 signals due to extraction errors.
>>> [extended] Skipped 105 signals due to extraction errors.
>>> [experimental] Skipped 105 signals due to extraction errors.
Original - core: (40, 5)  | extended: (40, 8)  | experimental: (40, 24)
Augmented -  core: (195, 5)   | extended: (195, 8)    | experimental: (195, 24)
Combined - core: (235, 5) | extended: (235, 8) | experimental: (235, 24)


## Persist Feature CSVs

In [67]:
## Persist Feature CSVos.makedirs('vectorized', exist_ok=True)

save_map = {
    'full_augmented_core':         aug_core,
    'full_augmented_extended':     aug_ext,
    'full_augmented_experimental': aug_exp,
    'full_combined_core':          combined_core,
    'full_combined_extended':      combined_ext,
    'full_combined_experimental':  combined_exp,
}

for name, df in save_map.items():
    path = f'vectorized/{name}.csv'
    df.to_csv(path, index=False)
    print(f"  Saved  {path}  ({df.shape[0]} rows × {df.shape[1]} cols)")

print("\nAll CSVs saved to vectorized/")
print()

  Saved  vectorized/full_augmented_core.csv  (195 rows × 5 cols)
  Saved  vectorized/full_augmented_extended.csv  (195 rows × 8 cols)
  Saved  vectorized/full_augmented_experimental.csv  (195 rows × 24 cols)
  Saved  vectorized/full_combined_core.csv  (235 rows × 5 cols)
  Saved  vectorized/full_combined_extended.csv  (235 rows × 8 cols)
  Saved  vectorized/full_combined_experimental.csv  (235 rows × 24 cols)

All CSVs saved to vectorized/



## Ablation Study -  Which Full-Range Decisions Actually Help?